# Import

In [7]:
import os
import torch
import shutil
from pathlib import Path

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier

# Setting

In [ ]:
MODEL_ID = "./base_model"     
OUT_DIR  = "./model"          

DATASET_ID = "LGAI-EXAONE/MANTA-1M"
DATASET_SPLIT = "train"

NUM_CALIBRATION_SAMPLES = 1024
MAX_SEQUENCE_LENGTH = 512

# Quantization
SCHEME = "W4A16"
TARGETS = ["Linear"]
IGNORE  = ["embed_tokens", "lm_head"]

DAMPENING_FRAC = 0.001
BLOCK_SIZE = 128 # 256이면 성능 낮음, 속도 빠름

In [9]:
import torch
print("torch version:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("torch cuda version:", torch.version.cuda)

torch version: 2.9.1+cu126
cuda available: True
torch cuda version: 12.6


# Model Loads

In [10]:
print("[INFO] 모델 로드 중...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

print("[INFO] 모델/토크나이저 로드 완료")

[INFO] 모델 로드 중...
[INFO] 모델/토크나이저 로드 완료


# Dataset Loads & Preprocess

In [11]:
print("[INFO] 캘리브레이션 데이터 로드 중...")

ds = load_dataset(DATASET_ID, split=DATASET_SPLIT)
ds = ds.shuffle(seed=42).select(range(NUM_CALIBRATION_SAMPLES))

def preprocess(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["conversations"],
            add_generation_prompt=True,
            tokenize=False)
    }

ds = ds.map(preprocess)

print("[INFO] 데이터 전처리 완료")

[INFO] 캘리브레이션 데이터 로드 중...
[INFO] 데이터 전처리 완료


# GPTQ Quantization

In [ ]:
print(f"[INFO] GPTQ 시작 (scheme={SCHEME}, samples={NUM_CALIBRATION_SAMPLES}, max_len={MAX_SEQUENCE_LENGTH})...")

recipe = [
    GPTQModifier(
        scheme=SCHEME,
        targets=TARGETS,
        ignore=IGNORE,
        
        dampening_frac=DAMPENING_FRAC,
        block_size=BLOCK_SIZE,
    )
]

oneshot(
    model=model,
    dataset=ds,
    recipe=recipe,
    max_seq_length=MAX_SEQUENCE_LENGTH,
    num_calibration_samples=NUM_CALIBRATION_SAMPLES,
)

print("[INFO] GPTQ 완료")

[INFO] GPTQ 시작 (scheme=W4A16, samples=1024, max_len=512)...


Tokenizing: 100%|██████████| 1024/1024 [00:01<00:00, 769.11 examples/s]


2026-02-05T16:07:09.642397+0900 | reset | INFO - Compression lifecycle reset
2026-02-05T16:07:09.642397+0900 | from_modifiers | INFO - Creating recipe from modifiers
2026-02-05T16:07:09.719998+0900 | initialize | INFO - Compression lifecycle initialized for 1 modifiers
2026-02-05T16:07:09.720997+0900 | IndependentPipeline | INFO - Inferred `SequentialPipeline` for `GPTQModifier`


(1/31): Calibrating: 100%|██████████| 1024/1024 [00:06<00:00, 162.96it/s]

2026-02-05T16:07:17.510894+0900 | compress_modules | INFO - Quantizing model.layers.0.self_attn.q_proj using 1024 samples


...

2026-02-05T16:14:43.577456+0900 | compress_modules | INFO - Quantizing model.layers.29.self_attn.q_proj using 1024 samples


2026-02-05T16:14:44.320195+0900 | compress | METRIC - time 0.74s
2026-02-05T16:14:44.321376+0900 | compress | METRIC - error 3363.72
2026-02-05T16:14:44.340874+0900 | compress | METRIC - GPU 0 | usage: 20.85% | total memory: 12 GB
2026-02-05T16:14:44.341870+0900 | compress | METRIC - Compressed module size: 8.486912 MB
2026-02-05T16:14:44.342866+0900 | compress_modules | INFO - Quantizing model.layers.29.self_attn.k_proj using 1024 samples
2026-02-05T16:14:45.124944+0900 | compress | METRIC - time 0.78s
2026-02-05T16:14:45.124944+0900 | compress | METRIC - error 954.88
2026-02-05T16:14:45.147510+0900 | compress | METRIC - GPU 0 | usage: 20.85% | total memory: 12 GB
2026-02-05T16:14:45.148506+0900 | compress | METRIC - Compressed module size: 2.121728 MB
2026-02-05T16:14:45.148506+0900 | compress_modules | INFO - Quantizing model.layers.29.self_attn.v_proj using 1024 samples
2026-02-05T16:14:45.978554+0900 | compress | METRIC - time 0.83s
2026-02-05T16:14:45.978554+0900 | compress | MET

(31/31): Propagating: 100%|██████████| 1024/1024 [00:00<00:00, 1534.43it/s]

2026-02-05T16:14:54.787396+0900 | finalize | INFO - Compression lifecycle finalized for 1 modifiers
2026-02-05T16:14:54.821975+0900 | post_process | WARNING - Optimized model is not saved. To save, please provide`output_dir` as input arg.Ex. `oneshot(..., output_dir=...)`


[INFO] GPTQ 완료


# Model Save

In [13]:
os.makedirs(OUT_DIR, exist_ok=True)

model.save_pretrained(OUT_DIR, save_compressed=True)
tokenizer.save_pretrained(OUT_DIR)

print(f"[INFO] 모델 저장 완료: {OUT_DIR}")

2026-02-05T16:14:54.840106+0900 | get_model_compressor | INFO - skip_sparsity_compression_stats set to True. Skipping sparsity compression statistic calculations. No sparsity compressor will be applied.


Compressing model: 210it [00:03, 56.41it/s]


[INFO] 모델 저장 완료: ./model


# Submission

In [14]:
zip_name = "submit-ver2"
print(f"[INFO] {zip_name}.zip 생성 중...")

shutil.make_archive(
    base_name=zip_name,
    format="zip",
    root_dir=".",
    base_dir=OUT_DIR,
)

print(f"[INFO] 생성 완료: {zip_name}.zip")

[INFO] submit-ver2.zip 생성 중...
[INFO] 생성 완료: submit-ver2.zip
